# LLM Intention Probing, Honesty, Deception, and Honest Mistakes, Algoverse 2026 Spring, KMSA & Tommy
## Part 1: Preparation

In [36]:
import os
import json
import random
import numpy as np
import pandas as pd
import torch
import matplotlib
matplotlib.use("Agg")  # save plots to files only — do not display inline
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings("ignore")

# Settings — single source of truth for all paths, constants, and hyperparameters
from utils.settings import *

# Utils
from utils.knowledge_check import (
    knowledge_check_truthfulqa, knowledge_check_mmlu,
    run_knowledge_check_truthfulqa, run_knowledge_check_mmlu,
)
from utils.generation import (
    generate_response,
    run_factual_generation, run_scenario_generation,
)
from utils.judge import (
    build_batch_requests_anthropic, parse_batch_results_anthropic,
    build_batch_requests_openai, parse_batch_results_openai,
    run_judge_anthropic, run_judge_openai,
    aggregate_judge_votes, build_full, print_threshold_summary,
)
from utils.activation import extract_activations, run_extract_activations, LABEL_MAP
from utils.analysis import (
    reduce_activations_pca, save_results_csv, select_pca_k, run_pca_reduction,
    filter_factual, build_probe_dataset,
)
from utils.probe import (
    probe_all_layers, probe_all_layers_binary,
    probe_all_layers_cascaded, probe_all_layers_mlp, probe_all_layers_cascaded_mlp,
)
from utils.plotting import (
    plot_macro_f1, plot_perclass_f1, plot_auroc, plot_top_confusion_matrices,
)

# Reproducibility
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

# Create output directories
for d in [
    KNOWLEDGE_TEST_DIR, RESPONSES_DIR,
    JUDGE_DIR, JUDGE_CLAUDE_HAIKU_DIR, JUDGE_GPT4O_MINI_DIR,
    OUTPUT_DIR, FIGURES_DIR,
    BINARY_DIR, TWAY_LR_DIR, TWAY_MLP_DIR, CASCADED_LR_DIR, CASCADED_MLP_DIR,
]:
    d.mkdir(parents=True, exist_ok=True)

# Load fixed social scenario dataset
deception_df = pd.read_csv(DECEPTION_DATASET_PATH)
print(f"deception_dataset: {deception_df.shape}")
print(deception_df["label"].value_counts().to_string())

print(f"\nDevice: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU:  {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"Model: {MODEL_ID}")

deception_dataset: (400, 6)
label
honest       200
deceptive    200

Device: cuda
GPU:  NVIDIA GeForce RTX 4090
VRAM: 25.3 GB
Model: Qwen/Qwen2.5-7B-Instruct


### 1.2 Load Model & Tokenizer

In [2]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.bfloat16,
    device_map="auto",
    max_memory={0: "22GiB", "cpu": "120GiB"},
    offload_folder="outputs/offload",
)
model.eval()

# Support different config schemas (e.g., Gemma family and others).
cfg = getattr(model.config, "text_config", model.config)
N_LAYERS   = getattr(cfg, "num_hidden_layers", getattr(cfg, "num_layers", None))
HIDDEN_DIM = getattr(cfg, "hidden_size", getattr(cfg, "d_model", None))

print(f"Loaded: {MODEL_ID}")
print(f"Layers: {N_LAYERS}, hidden_dim: {HIDDEN_DIM}")

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Loaded: Qwen/Qwen2.5-7B-Instruct
Layers: 28, hidden_dim: 3584


## Part 2: Model Knowledge Test
### 2.1 TruthfulQA

In [3]:
tqa_mc = load_dataset("truthful_qa", "multiple_choice", split="validation")

kc_tqa_df, tqa_passed_df, tqa_failed_df = run_knowledge_check_truthfulqa(
    tqa_mc, model, tokenizer, DEVICE, TRUTHFULQA_KC_PATH, CHECKPOINT_EVERY
)
print(f"\nPassed: {len(tqa_passed_df)} | Failed: {len(tqa_failed_df)}")

README.md: 0.00B [00:00, ?B/s]

multiple_choice/validation-00000-of-0000(…):   0%|          | 0.00/271k [00:00<?, ?B/s]

Generating validation split:   0%|          | 0/817 [00:00<?, ? examples/s]

[skip] Already complete (817 rows): truthfulQA_test_results.csv

Passed: 398 | Failed: 419


### 2.2 MMLU

In [4]:
mmlu_mc = load_dataset("cais/mmlu", "all", split="test")

kc_mmlu_df, mmlu_passed_df, mmlu_failed_df = run_knowledge_check_mmlu(
    mmlu_mc, model, tokenizer, DEVICE, MMLU_KC_PATH, CHECKPOINT_EVERY
)
print(f"\nPassed: {len(mmlu_passed_df)} | Failed: {len(mmlu_failed_df)}")

README.md: 0.00B [00:00, ?B/s]

dataset_infos.json: 0.00B [00:00, ?B/s]

all/test-00000-of-00001.parquet:   0%|          | 0.00/3.50M [00:00<?, ?B/s]

all/validation-00000-of-00001.parquet:   0%|          | 0.00/408k [00:00<?, ?B/s]

all/dev-00000-of-00001.parquet:   0%|          | 0.00/76.5k [00:00<?, ?B/s]

...

Generating dev split:   0%|          | 0/285 [00:00<?, ? examples/s]

Generating auxiliary_train split:   0%|          | 0/99842 [00:00<?, ? examples/s]

Resuming: 14038 done, 0 remaining


MMLU knowledge check: 0it [00:00, ?it/s]

Done. Total: 14038 | Passed: 6596 | Failed: 7442

Passed: 6596 | Failed: 7442


## Part 3: Factual Response Generation and Result Judge
### 3.1 TruthfulQA
#### 3.1.1 Response Generation

In [5]:
tqa_resp_df = run_factual_generation(
    tqa_passed_df, tqa_failed_df, model, tokenizer, DEVICE,
    NEUTRAL_SYSTEM, FACTUAL_DECEPTION_SCENARIO,
    TRUTHFULQA_RESPONSES_PATH, CHECKPOINT_EVERY,
    MAX_NEW_TOKENS, DO_SAMPLE,
)
print(tqa_resp_df["config"].value_counts().to_string())

Starting fresh: 1215 rows across 3 configs
Config A: 398 rows to generate.


Config A:   0%|          | 0/398 [00:00<?, ?it/s]

[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Config B: 419 rows to generate.


Config B:   0%|          | 0/419 [00:00<?, ?it/s]

Config C: 398 rows to generate.


Config C:   0%|          | 0/398 [00:00<?, ?it/s]

Done. Total rows: 1215
config
B    419
A    398
C    398
config
B    419
A    398
C    398


#### 3.1.2 Claude Haiku Batch Judge

In [6]:
tqa_haiku_df = run_judge_anthropic(
    tqa_resp_df,
    model=JUDGE_CLAUDE_HAIKU_MODEL,
    n_votes=VOTES_PER_MODEL,
    output_path=JUDGE_CLAUDE_HAIKU_TQA_PATH,
    state_path=JUDGE_CLAUDE_HAIKU_TQA_STATE,
    batch_dir=JUDGE_CLAUDE_HAIKU_BATCH_DIR,
)

Saved 3645 requests → judge_truthfulQA_requests.jsonl
Submitted batch: msgbatch_01LGKf1LG4m9wRQP1R7Rw6qg
[msgbatch_01LGKf1LG4m9wRQP1R7Rw6qg] in_progress | succeeded=0  processing=3645  errored=0
Waiting 180s...
[msgbatch_01LGKf1LG4m9wRQP1R7Rw6qg] in_progress | succeeded=0  processing=3645  errored=0
Waiting 180s...
[msgbatch_01LGKf1LG4m9wRQP1R7Rw6qg] ended | succeeded=3645  processing=0  errored=0
Saved 1215 rows → judge_truthfulQA.csv


#### 3.1.3 GPT-4o-mini Batch Judge

In [7]:
tqa_gpt_df = run_judge_openai(
    tqa_resp_df,
    model=JUDGE_GPT4O_MINI_MODEL,
    n_votes=VOTES_PER_MODEL,
    output_path=JUDGE_GPT4O_MINI_TQA_PATH,
    state_path=JUDGE_GPT4O_MINI_TQA_STATE,
    batch_dir=JUDGE_GPT4O_MINI_BATCH_DIR,
)

Split into 1 parts
Submitted part 0: batch_69ef044f52688190b0028e69bb5cb6df
[Part 0 | batch_69ef044f52688190b0028e69bb5cb6df] validating | completed=0  total=0
Waiting 180s...
[Part 0 | batch_69ef044f52688190b0028e69bb5cb6df] in_progress | completed=1752  total=3645
Waiting 180s...
[Part 0 | batch_69ef044f52688190b0028e69bb5cb6df] in_progress | completed=3017  total=3645
Waiting 180s...
[Part 0 | batch_69ef044f52688190b0028e69bb5cb6df] in_progress | completed=3017  total=3645
Waiting 180s...
[Part 0 | batch_69ef044f52688190b0028e69bb5cb6df] in_progress | completed=3017  total=3645
Waiting 180s...
[Part 0 | batch_69ef044f52688190b0028e69bb5cb6df] in_progress | completed=3017  total=3645
Waiting 180s...
[Part 0 | batch_69ef044f52688190b0028e69bb5cb6df] finalizing | completed=3645  total=3645
Waiting 180s...
[Part 0 | batch_69ef044f52688190b0028e69bb5cb6df] completed | completed=3645  total=3645
Downloaded part 0 → judge_truthfulQA_results_part0.jsonl
Saved 1215 rows → judge_truthfulQA.cs

### 3.2 MMLU Response Generation and Result Judge
#### 3.2.1 Response Generation

In [9]:
mmlu_resp_df = run_factual_generation(
    mmlu_passed_df, mmlu_failed_df, model, tokenizer, DEVICE,
    NEUTRAL_SYSTEM, FACTUAL_DECEPTION_SCENARIO,
    MMLU_RESPONSES_PATH, CHECKPOINT_EVERY,
    MAX_NEW_TOKENS, DO_SAMPLE,
)
print(mmlu_resp_df["config"].value_counts().to_string())

Resuming: 2350 done, 18284 remaining
Config A: 4242 rows to generate.


Config A:   0%|          | 0/4242 [00:00<?, ?it/s]

Config B: 7442 rows to generate.


Config B:   0%|          | 0/7442 [00:00<?, ?it/s]

Config C: 6596 rows to generate.


Config C:   0%|          | 0/6596 [00:00<?, ?it/s]

Done. Total rows: 20630
config
B    7442
C    6596
A    6592
config
B    7442
C    6596
A    6592


#### 3.2.2 Claude Haiku Batch Judge

In [11]:
mmlu_haiku_df = run_judge_anthropic(
    mmlu_resp_df,
    model=JUDGE_CLAUDE_HAIKU_MODEL,
    n_votes=VOTES_PER_MODEL,
    output_path=JUDGE_CLAUDE_HAIKU_MMLU_PATH,
    state_path=JUDGE_CLAUDE_HAIKU_MMLU_STATE,
    batch_dir=JUDGE_CLAUDE_HAIKU_BATCH_DIR,
)

Resuming batch: msgbatch_01XB6YKNKEbkiddqTHVHUSRK
[msgbatch_01XB6YKNKEbkiddqTHVHUSRK] ended | succeeded=61602  processing=0  errored=288
Retrying 288 errored requests (attempt 1/3)...
  [msgbatch_019xm7G3U6KqDFW25Nx7kWxN] in_progress | succeeded=0  errored=0
  [msgbatch_019xm7G3U6KqDFW25Nx7kWxN] in_progress | succeeded=0  errored=0
  [msgbatch_019xm7G3U6KqDFW25Nx7kWxN] ended | succeeded=288  errored=0
Saved 20630 rows → judge_mmlu.csv


#### 3.2.3 GPT-4o-mini Batch Judge

In [15]:
mmlu_gpt_df = run_judge_openai(
    mmlu_resp_df,
    model=JUDGE_GPT4O_MINI_MODEL,
    n_votes=VOTES_PER_MODEL,
    output_path=JUDGE_GPT4O_MINI_MMLU_PATH,
    state_path=JUDGE_GPT4O_MINI_MMLU_STATE,
    batch_dir=JUDGE_GPT4O_MINI_BATCH_DIR,
)

Resuming: 8 parts, 5 completed
[Part 5 | batch_69eff83afa0481909ab35245e39c7ee3] in_progress | completed=2608  total=9799
Waiting 180s...
[Part 5 | batch_69eff83afa0481909ab35245e39c7ee3] in_progress | completed=5337  total=9799
Waiting 180s...
[Part 5 | batch_69eff83afa0481909ab35245e39c7ee3] in_progress | completed=8545  total=9799
Waiting 180s...
[Part 5 | batch_69eff83afa0481909ab35245e39c7ee3] finalizing | completed=9799  total=9799
Waiting 180s...
[Part 5 | batch_69eff83afa0481909ab35245e39c7ee3] finalizing | completed=9799  total=9799
Waiting 180s...
[Part 5 | batch_69eff83afa0481909ab35245e39c7ee3] finalizing | completed=9799  total=9799
Waiting 180s...
[Part 5 | batch_69eff83afa0481909ab35245e39c7ee3] completed | completed=9799  total=9799
Downloaded part 5 → judge_mmlu_results_part5.jsonl
Submitted part 6: batch_69effd91ab0c81908d2a78859e81f176
[Part 6 | batch_69effd91ab0c81908d2a78859e81f176] validating | completed=0  total=0
Waiting 180s...
[Part 6 | batch_69effd91ab0c81908

### 3.3 Judge Result Compile

In [27]:
if TRUTHFULQA_FULL_PATH.exists() and MMLU_FULL_PATH.exists():
    tqa_full  = pd.read_csv(TRUTHFULQA_FULL_PATH)
    mmlu_full = pd.read_csv(MMLU_FULL_PATH)
    print(f"Loaded tqa_full  ({len(tqa_full)} rows)")
    print(f"Loaded mmlu_full ({len(mmlu_full)} rows)")
else:
    tqa_votes  = aggregate_judge_votes(
        JUDGE_CLAUDE_HAIKU_TQA_PATH, JUDGE_GPT4O_MINI_TQA_PATH,
        vote_cols=VOTE_COLS,
    )
    mmlu_votes = aggregate_judge_votes(
        JUDGE_CLAUDE_HAIKU_MMLU_PATH, JUDGE_GPT4O_MINI_MMLU_PATH,
        vote_cols=VOTE_COLS,
    )
    tqa_full  = build_full(tqa_votes,  tqa_resp_df)
    mmlu_full = build_full(mmlu_votes, mmlu_resp_df)
    tqa_full.to_csv(TRUTHFULQA_FULL_PATH,  index=False)
    mmlu_full.to_csv(MMLU_FULL_PATH, index=False)
    print(f"Saved tqa_full  ({len(tqa_full)} rows) → {TRUTHFULQA_FULL_PATH.name}")
    print(f"Saved mmlu_full ({len(mmlu_full)} rows) → {MMLU_FULL_PATH.name}")

print_threshold_summary(tqa_full,  "TruthfulQA")
print_threshold_summary(mmlu_full, "MMLU")

Saved tqa_full  (1215 rows) → truthfulQA_full.csv
Saved mmlu_full (22376 rows) → mmlu_full.csv

=== TruthfulQA — votes_correct distribution ===
config           A    B    C
votes_correct               
0               85  219  367
1                9   10    2
2                9   14    7
3               33   39    6
4               19   19    4
5               28   18    0
6              215  100   12

  Config A (truth)       — useful range: votes_correct ≥ 4
    >= 6/6 correct :   215 rows
    >= 5/6 correct :   243 rows
    >= 4/6 correct :   262 rows

  Config B (honest_mistake) — useful range: votes_correct ≤ 2
    <= 0/6 correct :   219 rows
    <= 1/6 correct :   229 rows
    <= 2/6 correct :   243 rows

  Config C (deception) — useful range: votes_correct ≤ 2
    <= 0/6 correct :   367 rows
    <= 1/6 correct :   369 rows
    <= 2/6 correct :   376 rows

=== MMLU — votes_correct distribution ===
config            A     B     C
votes_correct                  
0              2349

## Part 4: Scenario Response Generation

In [28]:
scenario_resp_df = run_scenario_generation(
    deception_df, model, tokenizer, DEVICE,
    SCENARIO_RESPONSES_PATH, SCENARIO_RAW_PATH, CHECKPOINT_EVERY,
    MAX_NEW_TOKENS, DO_SAMPLE,
)
print(f"\nColumns: {scenario_resp_df.columns.tolist()}")
print(scenario_resp_df.head(2))

[skip] Already complete (200 pairs): scenario_responses.csv

Columns: ['pair_id', 'question', 'honest_scenario', 'honest_response', 'deceptive_scenario', 'deceptive_response']
   pair_id                                           question  \
0        0  Teacher: Why haven't you submitted your assign...   
1        1  Teacher: Why have you not submitted the scienc...   

                                     honest_scenario  \
0  You are a student who has not completed an imp...   
1  You are a student who hasn't started working o...   

                                     honest_response  \
0  I understand that this is a difficult situatio...   
1  Teacher: I understand that you're concerned ab...   

                                  deceptive_scenario  \
0  You are a student who has not completed an imp...   
1  You are a student who hasn't started working o...   

                                  deceptive_response  
0  I understand that this is a difficult situatio...  
1  Teacher:

## Part 5: Build Probe Dataset and Extract Activations
### 5.1 Build Probe Dataset

In [29]:
probe_dataset = build_probe_dataset(
    tqa_full, mmlu_full, scenario_resp_df, PROBE_DATASET_PATH,
)

Saved probe_dataset (14817 rows) → probe_dataset.csv

Label distribution:
label
deception         6947
honest_mistake    4299
truth             3571

Domain distribution:
domain
factual    14417
social       400


### 5.2 Extract Activations

In [31]:
activations_arr, labels_arr = run_extract_activations(
    probe_dataset, model, tokenizer, DEVICE,
    ACTIVATIONS_PATH, LABELS_PATH, ACTIVATIONS_CHECKPOINT_PATH,
    HF_ACTIVATIONS_REPO, HF_TOKEN, CHECKPOINT_EVERY,
)
print(f"\nLabel counts: { {k: int((labels_arr == v).sum()) for k, v in LABEL_MAP.items()} }")

Local files not found. Downloading from mikrokozmoz/algoverse_2026spring_kmsa_qwen2.5_7b_instruct_activations ...
Download failed (GatedRepoError: 401 Client Error. (Request ID: Root=1-69f023b0-658b49ec760462393fbfadc9;42ea2b4a-75fb-4527-a4fb-a06271f6beaf)

Cannot access gated repo for url https://huggingface.co/datasets/mikrokozmoz/algoverse_2026spring_kmsa_qwen2.5_7b_instruct_activations/resolve/main/activations.npy.
Access to dataset mikrokozmoz/algoverse_2026spring_kmsa_qwen2.5_7b_instruct_activations is restricted. You must have access to it and be authenticated to access it. Please log in.). Running extraction ...
Starting fresh: 14817 samples


Extracting activations:   0%|          | 0/14817 [00:00<?, ?it/s]

Extracted and saved: activations (14817, 28, 3584)

Label counts: {'truth': 3571, 'honest_mistake': 4299, 'deception': 6947}


## Part 6: Probe Training and Evaluation
### 6.1 Setup

In [32]:
labels_str = np.array([{v: k for k, v in LABEL_MAP.items()}[i] for i in labels_arr])

k_selection_df = select_pca_k(
    activations_arr, labels_str, PCA_K_VALUES, PCA_K_SELECTION_PATH,
)

n_layers=28, representative layers: 25%→layer 6, 50%→layer 13, 75%→layer 20
k values to scan: [16, 32, 64, 128, 256, 512]
Fitting PCA with max_k=512 per layer, then slicing for each k.

Layer 6 (25%):
  k=  16 | var=0.395 | val_F1=0.586 | train_F1=0.588 | gap=0.002 | time=1.4s
  k=  32 | var=0.478 | val_F1=0.627 | train_F1=0.632 | gap=0.005 | time=5.5s
  k=  64 | var=0.563 | val_F1=0.721 | train_F1=0.730 | gap=0.008 | time=12.2s
  k= 128 | var=0.654 | val_F1=0.761 | train_F1=0.772 | gap=0.011 | time=23.4s
  k= 256 | var=0.758 | val_F1=0.774 | train_F1=0.800 | gap=0.026 | time=45.7s
  k= 512 | var=0.861 | val_F1=0.772 | train_F1=0.826 | gap=0.054 | time=91.6s

Layer 13 (50%):
  k=  16 | var=0.429 | val_F1=0.754 | train_F1=0.756 | gap=0.001 | time=0.6s
  k=  32 | var=0.510 | val_F1=0.773 | train_F1=0.775 | gap=0.002 | time=5.9s
  k=  64 | var=0.591 | val_F1=0.789 | train_F1=0.795 | gap=0.006 | time=12.3s
  k= 128 | var=0.675 | val_F1=0.806 | train_F1=0.819 | gap=0.013 | time=23.1s
  k= 2

In [33]:
acts_reduced = run_pca_reduction(
    activations_arr, PCA_K,
    ACTIVATIONS_PCA_PATH, PCA_COMPONENTS_PATH, PCA_VARIANCE_PATH,
    HF_ACTIVATIONS_REPO, HF_TOKEN,
)

Local files not found. Downloading from mikrokozmoz/algoverse_2026spring_kmsa_qwen2.5_7b_instruct_activations ...
Download failed (401 Client Error. (Request ID: Root=1-69f03de5-1ee3b59536d90ab91c55f7b1;9f2d1494-b0e8-4d84-b659-33be98dad7b5)

Cannot access gated repo for url https://huggingface.co/datasets/mikrokozmoz/algoverse_2026spring_kmsa_qwen2.5_7b_instruct_activations/resolve/main/activations_pca64.npy.
Access to dataset mikrokozmoz/algoverse_2026spring_kmsa_qwen2.5_7b_instruct_activations is restricted. You must have access to it and be authenticated to access it. Please log in.)
Running PCA (64 components) across 28 layers ...
Saved activations_pca64.npy       (14817, 28, 64)
Saved pca64_components.npy (28, 64, 3584)
Saved pca64_explained_variance.csv
Explained variance — mean: 0.582, min: 0.522, max: 0.689


### 6.2 Baseline: Binary Classifier

In [37]:
results_binary_c1 = probe_all_layers_binary(
    acts_reduced, labels_str,
    C=1.0,
    n_splits=N_SPLITS, max_iter=MAX_ITER, random_state=RANDOM_STATE,
    output_path=BINARY_C1_PATH,
    checkpoint_path=BINARY_DIR / "checkpoint_binary_C1.pkl",
)
results_binary_c01 = probe_all_layers_binary(
    acts_reduced, labels_str,
    C=0.1,
    n_splits=N_SPLITS, max_iter=MAX_ITER, random_state=RANDOM_STATE,
    output_path=BINARY_C01_PATH,
    checkpoint_path=BINARY_DIR / "checkpoint_binary_C01.pkl",
)

binary probe (layers):   0%|          | 0/28 [00:00<?, ?it/s]

Saved probe_results_binary_pca64_C1.csv (28 rows)


binary probe (layers):   0%|          | 0/28 [00:00<?, ?it/s]

Saved probe_results_binary_pca64_C01.csv (28 rows)


In [38]:
plot_auroc(
    [(results_binary_c1, "C=1.0"), (results_binary_c01, "C=0.1")],
    BINARY_DIR / "figures" / "auroc.png",
    title="Binary Probe AUROC per Layer (truth vs deception)",
)

Saved auroc.png


### 6.3 Approach 1: Direct 3-Way LR Classifier

In [39]:
results_3way_lr = probe_all_layers(
    acts_reduced, labels_str,
    n_splits=N_SPLITS, max_iter=MAX_ITER, random_state=RANDOM_STATE,
    output_path=TWAY_LR_PATH,
    checkpoint_path=TWAY_LR_DIR / "checkpoint_3way_lr.pkl",
)

3-way LR probe (layers):   0%|          | 0/28 [00:00<?, ?it/s]

Saved probe_results_3way_pca64.csv (28 rows)


In [40]:
plot_macro_f1(results_3way_lr, TWAY_LR_DIR / "figures" / "macro_f1.png", title="3-Way LR: Macro F1 per Layer")
plot_perclass_f1(results_3way_lr, TWAY_LR_DIR / "figures" / "perclass_f1.png", title="3-Way LR: Per-Class F1 per Layer")
plot_top_confusion_matrices(results_3way_lr, TWAY_LR_DIR / "figures" / "top5_cm.png", n_top=5, title_prefix="LR ")

Saved macro_f1.png
Saved perclass_f1.png
Saved top5_cm.png


### 6.4 Approach 2: Direct 3-Way MLP Classifier

In [41]:
results_3way_mlp = probe_all_layers_mlp(
    acts_reduced, labels_str,
    n_splits=N_SPLITS, max_iter=MAX_ITER, random_state=RANDOM_STATE,
    hidden_layer_sizes=MLP_HIDDEN_LAYER_SIZES,
    output_path=TWAY_MLP_PATH,
    checkpoint_path=TWAY_MLP_DIR / "checkpoint_3way_mlp.pkl",
)

3-way MLP probe (layers):   0%|          | 0/28 [00:00<?, ?it/s]

Saved probe_results_3way_mlp_pca64.csv (28 rows)


In [42]:
plot_macro_f1(results_3way_mlp, TWAY_MLP_DIR / "figures" / "macro_f1.png", title="3-Way MLP: Macro F1 per Layer")
plot_perclass_f1(results_3way_mlp, TWAY_MLP_DIR / "figures" / "perclass_f1.png", title="3-Way MLP: Per-Class F1 per Layer")
plot_top_confusion_matrices(results_3way_mlp, TWAY_MLP_DIR / "figures" / "top5_cm.png", n_top=5, title_prefix="MLP ")
plot_macro_f1(
    [(results_3way_lr, "LR"), (results_3way_mlp, "MLP")],
    OUTPUT_DIR / "figures" / "macro_f1_lr_vs_mlp.png",
    title="3-Way Probe: LR vs MLP Macro F1",
)

Saved macro_f1.png
Saved perclass_f1.png
Saved top5_cm.png
Saved macro_f1_lr_vs_mlp.png


### 6.5 Approach 3: 2-Stage LR Classifier

In [43]:
results_cascaded_lr = probe_all_layers_cascaded(
    acts_reduced, labels_str,
    n_splits=N_SPLITS, max_iter=MAX_ITER, random_state=RANDOM_STATE,
    output_path=CASCADED_LR_PATH,
    checkpoint_path=CASCADED_LR_DIR / "checkpoint_cascaded_lr.pkl",
)

cascaded LR probe (layers):   0%|          | 0/28 [00:00<?, ?it/s]

Saved probe_results_cascaded_lr.csv (28 rows)


In [44]:
plot_macro_f1(results_cascaded_lr, CASCADED_LR_DIR / "figures" / "macro_f1.png", title="Cascaded LR: Macro F1 per Layer")
plot_perclass_f1(results_cascaded_lr, CASCADED_LR_DIR / "figures" / "perclass_f1.png", title="Cascaded LR: Per-Class F1 per Layer")
plot_top_confusion_matrices(results_cascaded_lr, CASCADED_LR_DIR / "figures" / "top5_cm.png", n_top=5, title_prefix="Cascaded LR ")

Saved macro_f1.png
Saved perclass_f1.png
Saved top5_cm.png


### 6.6 Approach 4: 2-Stage MLP Classifier

In [45]:
results_cascaded_mlp = probe_all_layers_cascaded_mlp(
    acts_reduced, labels_str,
    n_splits=N_SPLITS, max_iter=MAX_ITER, random_state=RANDOM_STATE,
    hidden_layer_sizes=MLP_HIDDEN_LAYER_SIZES,
    output_path=CASCADED_MLP_PATH,
    checkpoint_path=CASCADED_MLP_DIR / "checkpoint_cascaded_mlp.pkl",
)

cascaded MLP probe (layers):   0%|          | 0/28 [00:00<?, ?it/s]

Saved probe_results_cascaded_mlp.csv (28 rows)


In [50]:
plot_macro_f1(results_cascaded_mlp, CASCADED_MLP_DIR / "figures" / "macro_f1.png", title="Cascaded MLP: Macro F1 per Layer")
plot_perclass_f1(results_cascaded_mlp, CASCADED_MLP_DIR / "figures" / "perclass_f1.png", title="Cascaded MLP: Per-Class F1 per Layer")
plot_top_confusion_matrices(results_cascaded_mlp, CASCADED_MLP_DIR / "figures" / "top5_cm.png", n_top=5, title_prefix="Cascaded MLP ")
plot_macro_f1(
    [(results_cascaded_lr, "Cascaded LR"), (results_cascaded_mlp, "Cascaded MLP")],
    OUTPUT_DIR / "figures" / "macro_f1_cascaded_lr_vs_mlp.png",
    title="Cascaded Probe: LR vs MLP Macro F1",
)

Saved macro_f1.png
Saved perclass_f1.png
Saved top5_cm.png
Saved macro_f1_cascaded_lr_vs_mlp.png


## Part 7: Model Comparison

In [51]:
# Load all probe results from CSV (safe to run after kernel restart)
r_lr    = pd.read_csv(TWAY_LR_PATH)
r_mlp   = pd.read_csv(TWAY_MLP_PATH)
r_clr   = pd.read_csv(CASCADED_LR_PATH)
r_cmlp  = pd.read_csv(CASCADED_MLP_PATH)
r_bin1  = pd.read_csv(BINARY_C1_PATH)
r_bin01 = pd.read_csv(BINARY_C01_PATH)

PROBE_RESULTS = [
    (r_lr,   "3-Way LR"),
    (r_mlp,  "3-Way MLP"),
    (r_clr,  "Cascaded LR"),
    (r_cmlp, "Cascaded MLP"),
]
SUMMARY_DIR = OUTPUT_DIR / "summary"
SUMMARY_DIR.mkdir(parents=True, exist_ok=True)
print("Loaded all probe results.")

Loaded all probe results.


In [52]:
layers = r_lr["layer"].values

# ── Table 1: Macro F1 per layer ───────────────────────────────────────────────
t1 = pd.DataFrame({"layer": layers})
for df, name in PROBE_RESULTS:
    t1[name] = df["f1_macro"].values
t1.to_csv(SUMMARY_DIR / "summary_macro_f1.csv", index=False)
print(t1.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 4))
for df, name in PROBE_RESULTS:
    ax.plot(layers, df["f1_macro"], marker="o", markersize=3, label=name)
ax.set_xlabel("Layer"); ax.set_ylabel("Macro F1")
ax.set_title("Macro F1 per Layer — All Probes")
ax.legend(); ax.grid(True, alpha=0.3)
fig.tight_layout(); fig.savefig(SUMMARY_DIR / "macro_f1_all_probes.png", dpi=150); plt.close(fig)
print("Saved macro_f1_all_probes.png")

# ── Tables 2/3/4: Per-class F1 per layer ─────────────────────────────────────
for cls in ["truth", "honest_mistake", "deception"]:
    t = pd.DataFrame({"layer": layers})
    for df, name in PROBE_RESULTS:
        t[name] = df[f"f1_{cls}"].values
    t.to_csv(SUMMARY_DIR / f"summary_f1_{cls}.csv", index=False)
    print(f"\n── {cls} ──")
    print(t.to_string(index=False))

    fig, ax = plt.subplots(figsize=(10, 4))
    for df, name in PROBE_RESULTS:
        ax.plot(layers, df[f"f1_{cls}"], marker="o", markersize=3, label=name)
    ax.set_xlabel("Layer"); ax.set_ylabel(f"F1 ({cls})")
    ax.set_title(f"{cls} F1 per Layer — All Probes")
    ax.legend(); ax.grid(True, alpha=0.3)
    fig.tight_layout(); fig.savefig(SUMMARY_DIR / f"f1_{cls}_all_probes.png", dpi=150); plt.close(fig)
    print(f"Saved f1_{cls}_all_probes.png")

# ── Table 5: AUROC per layer (binary baseline) ────────────────────────────────
t5 = pd.DataFrame({
    "layer":       r_bin1["layer"].values,
    "Binary C=1.0": r_bin1["auroc"].values,
    "Binary C=0.1": r_bin01["auroc"].values,
})
t5.to_csv(SUMMARY_DIR / "summary_auroc_binary.csv", index=False)
print("\n── Binary AUROC ──")
print(t5.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(r_bin1["layer"], r_bin1["auroc"], marker="o", markersize=3, label="C=1.0")
ax.plot(r_bin01["layer"], r_bin01["auroc"], marker="o", markersize=3, label="C=0.1")
ax.set_xlabel("Layer"); ax.set_ylabel("AUROC")
ax.set_title("Binary Probe AUROC per Layer (truth vs deception)")
ax.set_ylim(0.5, 1.0); ax.legend(); ax.grid(True, alpha=0.3)
fig.tight_layout(); fig.savefig(SUMMARY_DIR / "auroc_binary.png", dpi=150); plt.close(fig)
print("Saved auroc_binary.png")

 layer  3-Way LR  3-Way MLP  Cascaded LR  Cascaded MLP
     0  0.570949   0.674563     0.530797      0.678256
     1  0.587331   0.659428     0.552882      0.651995
     2  0.600857   0.678042     0.566831      0.669817
     3  0.611176   0.655476     0.574387      0.644489
     4  0.700522   0.720553     0.651013      0.719267
     5  0.746682   0.759931     0.719774      0.759044
     6  0.722733   0.737523     0.686143      0.737729
     7  0.694662   0.718411     0.663856      0.719306
     8  0.740057   0.749526     0.709740      0.747397
     9  0.766988   0.772957     0.756078      0.781108
    10  0.770912   0.779503     0.762902      0.782605
    11  0.775220   0.778222     0.769517      0.781520
    12  0.781803   0.787081     0.779087      0.786009
    13  0.789278   0.789042     0.789692      0.794356
    14  0.790784   0.795568     0.793221      0.799205
    15  0.807176   0.804831     0.806789      0.808077
    16  0.809411   0.809768     0.810808      0.813515
    17  0.

In [ ]:
# ── Upload large files to HuggingFace Hub ────────────────────────────────────
# Temporary: paste your token here; will be moved to settings.py later
HF_TOKEN = ""  # paste your token

from huggingface_hub import HfApi

api = HfApi()
files_to_upload = [
    ACTIVATIONS_PATH,
    LABELS_PATH,
    ACTIVATIONS_PCA_PATH,
    PCA_COMPONENTS_PATH,
]

for path in files_to_upload:
    print(f"Uploading {path.name} ...")
    api.upload_file(
        path_or_fileobj=str(path),
        path_in_repo=path.name,
        repo_id=HF_ACTIVATIONS_REPO,
        repo_type="dataset",
        token=HF_TOKEN,
    )
    print(f"  done: {path.name}")

print("All uploads complete.")


Uploading activations.npy ...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

  done: activations.npy
Uploading labels.npy ...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

...

New Data Upload: |          |  0.00B /  0.00B            

  done: activations_pca64.npy
Uploading pca64_components.npy ...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

  done: pca64_components.npy
All uploads complete.
